In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

print(catalog)

In [0]:
# --- Create ops.checkpoint_counts table if not exists ---
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_counts (
    checkpoint_name STRING,
    table_name      STRING,
    row_count       BIGINT,
    captured_at     TIMESTAMP
) USING DELTA
""")

# --- Run 1: AFTER_BATCH_1 — capture row counts for all tables ---
# 1 Batch 1 files only -- AFTER_BATCH_1
# 2 Upload all Batch 2 files  -- AFTER_BATCH_2
# 3 No new files -- AFTER_RERUN

checkpoint_name = "AFTER_RERUN"

# spark.sql(f"""
# INSERT INTO {catalog}.ops.checkpoint_counts
# SELECT '{checkpoint_name}', t.table_name, t.row_count, current_timestamp() AS captured_at
# FROM (
#     SELECT 'bronze.sessions'              AS table_name, COUNT(*) AS row_count FROM {catalog}.bronze.sessions
#     UNION ALL
#     SELECT 'bronze.members'              AS table_name, COUNT(*) AS row_count FROM {catalog}.bronze.member
#     UNION ALL
#     SELECT 'bronze.chargers',               COUNT(*) FROM {catalog}.bronze.chargers
#     UNION ALL
#     SELECT 'silver.sessions',               COUNT(*) FROM {catalog}.silver.sessions
#     UNION ALL
#     SELECT 'silver.quarantine_sessions',    COUNT(*) FROM {catalog}.silver.quarantine_sessions
#     UNION ALL
#     SELECT 'silver.members',                COUNT(*) FROM {catalog}.silver.members
#     UNION ALL
#     SELECT 'silver.chargers',               COUNT(*) FROM {catalog}.silver.chargers
#     UNION ALL
#     SELECT 'gold.fact_sessions',            COUNT(*) FROM {catalog}.gold.fact_sessions
#     UNION ALL
#     SELECT 'gold.agg_daily_station',        COUNT(*) FROM {catalog}.gold.agg_daily_station
# ) t
# """)



In [0]:
display(spark.sql(f"""
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY table_name ORDER BY table_name DESC) AS rn
    FROM {catalog}.ops.checkpoint_counts
)
-- WHERE rn = 3
ORDER BY captured_at DESC
"""))

##Job preview

![image_1792062469386.png](./image_1792062469386.png "image_1792062469386.png")